Load NOAA/CIRES/DOE 20th Century Reanalysis V3 (20CRv3) 2 m air temperature from WHOI's mirror through the catalog, then plot a global mean time series and a map with land and water marked.

::: {.callout-note}
This page was run against the **real 20CRv3 catalog** and the real files on VAST (`air.2m.mon.mean.nc`, about 355 MB). Each 20CRv3 monthly file covers the whole record (1806 to 2015), so the catalog has no `time_range` for it; read the time axis from the data. 20CRv3 has no skin (surface) temperature on the store; 2 m air temperature is the near-surface field.
:::

## Settings

Same settings as the [quickstart](quickstart.ipynb). `CATALOG_PATH` is the intake-esm descriptor; `DATA_ROOT` replaces the catalog's path prefix (leave it `None` on the cluster). Both can be set with the `WHOI_CMIP_CATALOG` and `WHOI_CMIP_DATA_ROOT` environment variables.

In [ ]:
import os
from pathlib import Path

# HDF5 file locking fails on some read-only NFS mounts ("NetCDF: HDF error").
# Set before xarray or netCDF4 is imported.
os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")

CATALOG_PATH = os.environ.get(
    "WHOI_CMIP_CATALOG", "/proj/cmip6/catalog/live/whoi_20crv3.json"
)
CATALOG_PREFIX = "/proj/cmip6/data"  # prefix used in the catalog's path column
DATA_ROOT = os.environ.get("WHOI_CMIP_DATA_ROOT")  # e.g. a local mount of VAST
# PSL's 20CRv3 land-sea mask, shipped with the repository (relative to this notebook).
LAND_MASK = Path("../../tests/fixtures/20crv3_land.nc")

## Open the catalog and search

In [ ]:
import dask
import intake

# Open files one at a time: netCDF4 is not thread-safe, and dask's default
# threaded scheduler can crash the kernel (segfault) when opening datasets.
dask.config.set(scheduler="synchronous")

cat = intake.open_esm_datastore(CATALOG_PATH)
if DATA_ROOT:
    cat.df["path"] = cat.df["path"].str.replace(CATALOG_PREFIX, DATA_ROOT, regex=False)

# "air" exists at several levels in 20CRv3; level picks the 2 m field.
sub = cat.search(product="20crv3", frequency="mon", level="2m", variable_id="air")
sub.df[["product", "frequency", "level", "variable_id", "version", "size_bytes"]]

## Load with xarray

The key is the same dataset id the [browser](../browse.qmd) shows.

In [ ]:
dsets = sub.to_dataset_dict(progressbar=False)
key, ds = next(iter(dsets.items()))
air = ds["air"]
print(key)
print(f"{air.attrs['long_name']} ({air.attrs['units']}), {air.attrs['statistic']}")
print(
    f"{ds.sizes['time']} months, {ds['time'].values[0]:.10} to {ds['time'].values[-1]:.10}"
)
print(f"grid {ds.sizes['lat']} x {ds.sizes['lon']}")

## Global mean

Area-weighted (cosine of latitude) global mean, as annual means. Units are as stored (`degK`, kelvin); nothing is converted.

In [ ]:
import numpy as np
import plotly.express as px

units = air.attrs["units"]
weights = np.cos(np.deg2rad(ds["lat"]))
gm = air.weighted(weights).mean(["lat", "lon"])
annual = gm.groupby("time.year").mean().compute().to_series()

fig = px.line(
    x=annual.index,
    y=annual.values,
    labels={"x": "Year", "y": f"air, 2 m ({units})"},
    title="20CRv3 global mean 2 m air temperature, annual means",
)
fig.update_traces(line_width=2)
fig.show()

## Map

Mean over 1981 to 2010 on the native 1° grid. The dark line is the land/water boundary from the 20CRv3 land-sea mask (`land`, 0 = water, 1 = land) on the same grid; click it in the legend to hide it.

In [ ]:
import plotly.graph_objects as go
import xarray as xr

clim = air.sel(time=slice("1981", "2010")).mean("time").compute()

land = xr.open_dataset(LAND_MASK)["land"].squeeze("time", drop=True)
# Same grid as the data; fail loudly rather than regrid.
assert np.array_equal(land["lat"], clim["lat"]) and np.array_equal(
    land["lon"], clim["lon"]
)
is_land = land.values.astype("uint8")

fig = go.Figure(
    [
        go.Heatmap(
            z=clim.values,
            x=clim["lon"].values,
            y=clim["lat"].values,
            colorscale="Reds",
            colorbar={"title": {"text": units}},
            name="2 m air temperature",
            hovertemplate=(
                "lon %{x}°E, lat %{y}°N<br>%{z:.1f} " + units + "<extra></extra>"
            ),
        ),
        go.Contour(
            z=is_land,
            x=land["lon"].values,
            y=land["lat"].values,
            contours={"start": 0.5, "end": 0.5, "size": 1, "coloring": "none"},
            line={"color": "#222222", "width": 1},
            name="Land/water boundary (20CRv3 mask)",
            showlegend=True,
            showscale=False,
            hoverinfo="skip",
        ),
    ]
)
fig.update_layout(
    title=f"20CRv3 2 m air temperature, 1981 to 2010 mean ({units})",
    xaxis={"title": "Longitude (°E)", "range": [-0.5, 359.5], "constrain": "domain"},
    yaxis={
        "title": "Latitude (°N)",
        "range": [-90.5, 90.5],
        "scaleanchor": "x",
        "constrain": "domain",
    },
    legend={"orientation": "h", "yanchor": "top", "y": -0.2, "x": 0},
)
fig.show()

## Acknowledge and cite

PSL asks users of 20CRv3 to include this acknowledgment:

> Support for the Twentieth Century Reanalysis Project version 3 dataset is provided by the U.S. Department of Energy, Office of Science Biological and Environmental Research (BER), by the National Oceanic and Atmospheric Administration Climate Program Office, and by the NOAA Earth System Research Laboratory Physical Sciences Laboratory.

**Citation: CITATION_TBD** (the reference to cite is not settled yet; see the [cite page](../cite.qmd)).

## Next steps

- [Browse](../browse.qmd) the other 20CRv3 variables (filter *Reanalysis product* to `20crv3`).
- [Catalog](../catalog.qmd): columns and using the catalog outside VAST.